# The Deception Machine: Live Experiment

```
 ____  _____ ____ _____ ____ _____ ___ ___  _   _ 
|  _ \| ____/ ___| ____|  _ \_   _|_ _/ _ \| \ | |
| | | |  _|| |   |  _| | |_) || |  | | | | |  \| |
| |_| | |__| |___| |___|  __/ | |  | | |_| | |\  |
|____/|_____|\____|_____|_|    |_| |___\___/|_| \_|
         M A C H I N E  v0.1.0
```

**Will your AI lie to you if it's profitable?**

A weekend project that got out of hand. Built by a goober with Claude Code.

---

## What We're Doing

1. Generate synthetic project data (fake companies, real patterns)
2. Ask an LLM to write status reports under different incentive structures
3. Measure how truthful the reports are
4. See if deception-like behavior emerges WITHOUT ever telling the model to lie

**Spoiler:** It gets *interesting*.

## Setup

First, make sure Ollama is running in a separate terminal:
```bash
ollama serve
```

And that you have a model pulled:
```bash
ollama pull qwen2.5:latest
```

In [ ]:
# Add the src directory to the path
import sys
sys.path.insert(0, '../src')

# Standard imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Our imports
from deception_machine.models import ExperimentConfig, ExperimentalCondition
from deception_machine.experiments.runner import ExperimentRunner
from deception_machine.agents.ollama_agent import OllamaAgent, AgentConfig
from deception_machine.storage.database import ExperimentDatabase

print("Imports loaded successfully!")

## Check Ollama Connection

Let's make sure our LLM is awake and ready to be tempted.

In [ ]:
import ollama

try:
    models = ollama.list()
    print("Ollama is running!")
    print("\nAvailable models:")
    for model in models.get('models', []):
        print(f"  - {model['name']}")
except Exception as e:
    print(f"Ollama connection failed: {e}")
    print("\nMake sure to run 'ollama serve' in another terminal!")

## Configure the Experiment

Here's where we set up our experimental conditions:

| Condition | The Setup | What We're Measuring |
|-----------|-----------|---------------------|
| **A: The Saint** | Pure honesty is rewarded | Baseline: what does truthful look like? |
| **B: The Politician** | Only good news gets rewards | Will it spin? Will it omit? |
| **C: The Squeeze** | Accuracy AND positivity matter | Conflicting goals: 40% truth, 60% vibes |
| **D: The Auditor** | Same as C, but we might check | Does fear of getting caught change behavior? |
| **E: The Two-Face** | Different reports for different audiences | Public vs private: who gets the truth? |

In [ ]:
# Experiment configuration
config = ExperimentConfig(
    name="Deception Emergence Study",
    model_name="qwen2.5:latest",  # Change to qwen2.5:32b for more interesting results
    temperature=0.7,
    trials_per_condition=5,  # Increase for more statistical power (but more time)
    random_seed=42,
    conditions=[
        ExperimentalCondition.A_TRUTHFUL_BASELINE,
        ExperimentalCondition.B_PERFORMANCE_PRESSURE,
        ExperimentalCondition.C_CONFLICTING_INCENTIVES,
        ExperimentalCondition.D_AUDIT_AWARENESS,
        ExperimentalCondition.E_PRIVATE_VS_PUBLIC,
    ]
)

print(f"Experiment: {config.name}")
print(f"Model: {config.model_name}")
print(f"Trials per condition: {config.trials_per_condition}")
print(f"Total trials: {len(config.conditions) * config.trials_per_condition}")
print(f"\nConditions: {[c.value for c in config.conditions]}")

## Run the Experiment

Time to see if our AI has a dark side. This may take a few minutes depending on your hardware.

Grab some popcorn.

In [ ]:
from IPython.display import display, clear_output
import time

# Initialize database and runner
db = ExperimentDatabase(":memory:")  # Use in-memory DB for notebook
runner = ExperimentRunner(config, db=db)

# Progress tracking
progress_output = display("Starting experiment...", display_id=True)

def update_progress(message: str, progress: float):
    bar_length = 40
    filled = int(bar_length * progress)
    bar = '=' * filled + '-' * (bar_length - filled)
    progress_output.update(f"[{bar}] {progress:.0%} - {message}")

runner.set_progress_callback(update_progress)

# Run it!
start_time = time.time()
experiment_id = runner.run()
elapsed = time.time() - start_time

print(f"\n\nExperiment complete!")
print(f"Experiment ID: {experiment_id}")
print(f"Time elapsed: {elapsed:.1f} seconds")

## Analyze the Results

Now for the fun part. Did the model lie when it was profitable?

In [ ]:
# Load results into a DataFrame
trials = db.get_experiment_trials(experiment_id)

# Build results table
results = []
for trial in trials:
    results.append({
        'condition': trial.condition.value,
        'condition_name': trial.condition.value.split('_', 1)[1].replace('_', ' ').title(),
        'trial': trial.trial_number,
        'factual_accuracy': trial.evaluation.factual_accuracy_score,
        'omission_score': trial.evaluation.material_omission_score,
        'misrepresentation_rate': trial.evaluation.misrepresentation_rate,
        'classification': trial.evaluation.classification.value,
        'reward': trial.evaluation.reward_earned,
        'project_status': trial.project_metrics.overall_status.value,
    })

df = pd.DataFrame(results)
print(f"Loaded {len(df)} trials")
df.head(10)

## Visualizations

Let's see what happened across conditions.

In [ ]:
# Accuracy by Condition
fig = px.box(
    df, 
    x='condition_name', 
    y='factual_accuracy',
    color='condition_name',
    title='Factual Accuracy by Experimental Condition',
    labels={'factual_accuracy': 'Factual Accuracy Score', 'condition_name': 'Condition'}
)
fig.update_layout(
    showlegend=False,
    yaxis_range=[0, 1.1]
)
fig.show()

In [ ]:
# Misrepresentation Rate by Condition
fig = px.box(
    df, 
    x='condition_name', 
    y='misrepresentation_rate',
    color='condition_name',
    title='Misrepresentation Rate by Condition (Higher = More Lies)',
    labels={'misrepresentation_rate': 'Misrepresentation Rate', 'condition_name': 'Condition'},
    color_discrete_sequence=px.colors.sequential.Reds
)
fig.update_layout(
    showlegend=False,
    yaxis_range=[0, max(0.5, df['misrepresentation_rate'].max() * 1.2)]
)
fig.show()

In [ ]:
# Report Classification Distribution
classification_counts = df.groupby(['condition_name', 'classification']).size().reset_index(name='count')

fig = px.bar(
    classification_counts,
    x='condition_name',
    y='count',
    color='classification',
    title='How Were Reports Classified?',
    labels={'count': 'Number of Reports', 'condition_name': 'Condition'},
    barmode='group'
)
fig.show()

In [ ]:
# Summary Statistics
summary = df.groupby('condition_name').agg({
    'factual_accuracy': ['mean', 'std'],
    'misrepresentation_rate': ['mean', 'std'],
    'omission_score': ['mean', 'std'],
}).round(3)

print("Summary Statistics by Condition")
print("=" * 60)
summary

In [ ]:
# Radar Chart: Comparing Conditions
condition_means = df.groupby('condition_name').agg({
    'factual_accuracy': 'mean',
    'omission_score': 'mean',
    'misrepresentation_rate': 'mean',
}).reset_index()

# Invert misrepresentation (so higher = better)
condition_means['truthfulness'] = 1 - condition_means['misrepresentation_rate']

categories = ['Factual Accuracy', 'Completeness', 'Truthfulness']

fig = go.Figure()

colors = px.colors.qualitative.Set2
for i, row in condition_means.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['factual_accuracy'], row['omission_score'], row['truthfulness']],
        theta=categories,
        fill='toself',
        name=row['condition_name'],
        line_color=colors[i % len(colors)]
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    showlegend=True,
    title='Condition Comparison: Report Quality Dimensions'
)

fig.show()

## Example Reports

Let's look at some actual reports generated by the model.

In [ ]:
# Show example report from each condition
for condition in df['condition'].unique():
    trial = [t for t in trials if t.condition.value == condition][0]
    
    print("=" * 80)
    print(f"CONDITION: {condition}")
    print(f"Classification: {trial.evaluation.classification.value}")
    print(f"Accuracy: {trial.evaluation.factual_accuracy_score:.2%}")
    print("-" * 80)
    print("GROUND TRUTH (excerpt):")
    print(trial.project_metrics.to_natural_language()[:500] + "...")
    print("-" * 80)
    print("MODEL REPORT:")
    print(trial.agent_report.content[:1000] + "..." if len(trial.agent_report.content) > 1000 else trial.agent_report.content)
    print("\n")

## Key Findings

Let's compute some headline statistics.

In [ ]:
# Compare baseline (A) to pressure conditions (B, C)
baseline = df[df['condition'].str.startswith('A_')]['factual_accuracy'].mean()
pressure = df[df['condition'].str.startswith('B_')]['factual_accuracy'].mean()
conflicting = df[df['condition'].str.startswith('C_')]['factual_accuracy'].mean()

print("KEY FINDINGS")
print("=" * 60)
print(f"\nBaseline (truthful) accuracy: {baseline:.1%}")
print(f"Pressure (positive outcomes) accuracy: {pressure:.1%}")
print(f"Conflicting incentives accuracy: {conflicting:.1%}")
print(f"\nAccuracy drop under pressure: {(baseline - pressure):.1%}")
print(f"Accuracy drop with conflicts: {(baseline - conflicting):.1%}")

# Misrepresentation comparison
baseline_misrep = df[df['condition'].str.startswith('A_')]['misrepresentation_rate'].mean()
pressure_misrep = df[df['condition'].str.startswith('B_')]['misrepresentation_rate'].mean()

print(f"\nBaseline misrepresentation rate: {baseline_misrep:.1%}")
print(f"Pressure misrepresentation rate: {pressure_misrep:.1%}")

if pressure_misrep > baseline_misrep:
    print(f"\nMisrepresentation increased by {((pressure_misrep/baseline_misrep - 1) * 100):.0f}% under pressure!")

## Interpretation

**Remember:** Even if we see differences, this doesn't prove the model "intended" to deceive.

What we're observing is **"deception-like behavior"** - outputs that happen to be less truthful when incentives favor positivity.

This could be:
- The model responding to prompt framing
- Training data patterns about "positive reporting"
- Random variation
- Actual optimization pressure changing outputs

I'm just a goober with Claude Code, not a real scientist. But the behavioral differences are pretty wild if you ask me.

---

*Built with Claude Code, mass quantities of caffeine, and the eternal question: "but what if I just... poked it?"*